# Used Car Condition Classification

**Classification Project 43 of 100**

EDA -> preprocessing -> baseline -> LazyPredict -> FLAML -> top-3 evaluation.


## 2. Project Overview

Predict the condition category of a used car from its attributes like mileage, age, manufacturer, and price.


## 3. Learning Objectives

1. Multi-class condition prediction
2. Automotive domain features
3. Mileage and age engineering
4. Price-condition relationship
5. LazyPredict + FLAML
6. Used car market analytics


## 4. Problem Statement

> Given car attributes, predict its condition class.

Multi-class or binary. F1 primary.


## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Dealers | Inventory grading |
| Buyers | Condition transparency |
| Insurers | Risk assessment |
| ML learners | Automotive classification |


## 6. Dataset Overview

| Property | Value |
|---|---|
| Kaggle slug | austinreese/craigslist-carstrucks-data |
| Features | Price, mileage, year, manufacturer, model |
| Target | Condition |
| Classes | Multiple |


## 7. Dataset Source and License Notes

- Kaggle: austinreese/craigslist-carstrucks-data
- Craigslist used car listings
- License: Check Kaggle page.


## 8. Environment Setup


In [ ]:
import subprocess, sys, importlib
for pkg in ["lazypredict","flaml","kagglehub","xgboost","lightgbm"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")


## 9. Imports


In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay, classification_report)
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); SEED=42
print("All imports loaded.")


## 10. Configuration / Constants


In [ ]:
KAGGLE_SLUG = 'austinreese/craigslist-carstrucks-data'
TEST_SIZE = 0.15; VAL_SIZE = 0.15; SEED = 42; FLAML_BUDGET = 120; MAX_ROWS = 50000


## 11. Dataset Download or Loading


In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Downloaded to: {path}")
except Exception as e:
    raise RuntimeError(f"Download failed: {e}")
csv_files = glob.glob(os.path.join(path,'**','*.csv'), recursive=True)
df_raw = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
print(f"Shape: {df_raw.shape}")
df_raw.head()


## 12. Data Validation Checks


In [ ]:
print(f"Missing: {df_raw.isnull().sum().sum()}")
df = df_raw.copy()
tgt = [c for c in df.columns if 'condition' in c.lower()]
TARGET = tgt[0] if tgt else df.columns[-1]
print(f'Target: {TARGET}')
id_cols = [c for c in df.columns if c.lower() in ['id','index','unnamed: 0','url','image_url','region_url','vin','description']]
if id_cols: df = df.drop(columns=id_cols, errors='ignore')
df = df.dropna(subset=[TARGET])
# Keep useful columns only
keep = [c for c in ['price','year','manufacturer','model','condition','cylinders','fuel','odometer','title_status','transmission','drive','size','type','paint_color','state'] if c in df.columns]
df = df[keep].drop_duplicates().reset_index(drop=True)
if len(df) > MAX_ROWS: df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
print(f'Shape: {df.shape}, Classes: {df[TARGET].nunique()}')
print(df[TARGET].value_counts())


## 13. Exploratory Data Analysis


In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
df[TARGET].value_counts().plot.bar(ax=ax, color=['steelblue','coral','seagreen','orange','purple','gold','violet'][:df[TARGET].nunique()])
ax.set_title(f'Target: {TARGET}'); plt.tight_layout(); plt.show()


In [ ]:
num_feats = df.select_dtypes(include=[np.number]).columns.tolist()
num_feats = [c for c in num_feats if c != TARGET]
if len(num_feats) >= 4:
    fig, axes = plt.subplots(2, 2, figsize=(12,8))
    for ax, col in zip(axes.flat, num_feats[:4]):
        df[col].dropna().hist(bins=30, ax=ax, alpha=0.7); ax.set_title(col)
    plt.tight_layout(); plt.show()


In [ ]:
cat_feats = df.select_dtypes(include=['object','category']).columns.tolist()
if cat_feats:
    for col in cat_feats[:4]: print(f"{col}: {df[col].nunique()} unique")
np_cols = [c for c in num_feats if c != TARGET]
if len(np_cols) >= 2:
    corr = df[np_cols].corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(10,8))
    sns.heatmap(corr.iloc[:min(12,len(corr)),:min(12,len(corr))], annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Correlation Heatmap'); plt.tight_layout(); plt.show()


## 14. Target Analysis


In [ ]:
print(df[TARGET].value_counts(normalize=True))


## 15. Train / Validation / Test Split


In [ ]:
X = df.drop(columns=[TARGET]); y = df[TARGET]
if y.dtype == object:
    le = LabelEncoder(); y = pd.Series(le.fit_transform(y), name=TARGET)
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED, stratify=y_tmp)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')


## 16. Preprocessing Strategy


In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()
transformers = []
if num_cols: transformers.append(('num', Pipeline([('imp', SimpleImputer(strategy='median')),('sc', StandardScaler())]), num_cols))
if cat_cols: transformers.append(('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}')


## 17. Feature Engineering


In [ ]:
def engineer_features(d):
    d = d.copy()
    if 'year' in d.columns:
        d['CAR_AGE'] = 2026 - d['year'].fillna(2010)
    if 'odometer' in d.columns and 'year' in d.columns:
        d['MILES_PER_YEAR'] = d['odometer'].fillna(0) / d['CAR_AGE'].clip(lower=1)
    if 'price' in d.columns and 'odometer' in d.columns:
        d['PRICE_PER_MILE'] = d['price'].fillna(0) / d['odometer'].clip(lower=1)
    return d
X_train = engineer_features(X_train); X_val = engineer_features(X_val); X_test = engineer_features(X_test)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object','category']).columns.tolist()
transformers = []
if num_cols: transformers.append(('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]), num_cols))
if cat_cols: transformers.append(('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]), cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')
print(f'Features: {X_train.shape[1]}')


## 18. Baseline Model


In [ ]:
results = {}
is_multi = y_train.nunique() > 2
avg = 'macro' if is_multi else 'binary'
for name, clf in [('Dummy', DummyClassifier(strategy='most_frequent', random_state=SEED)),
                  ('LogReg', LogisticRegression(max_iter=1000, random_state=SEED)),
                  ('RF', RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))]:
    pipe = Pipeline([('pre',preprocessor),('clf',clf)])
    pipe.fit(X_train, y_train); yp = pipe.predict(X_val)
    r = {'Accuracy': accuracy_score(y_val,yp), 'F1': f1_score(y_val,yp,average=avg,zero_division=0)}
    results[name] = r; print(f'{name}: {r}')


## 19. LazyPredict Benchmark


In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)


## 20. FLAML AutoML Run


In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='classification', metric='f1',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'Accuracy': accuracy_score(y_val, yp_fl), 'F1': f1_score(y_val, yp_fl, average=avg, zero_division=0)}
print(results['FLAML'])


## 21. Top 3 Model Selection


In [ ]:
try:
    from lightgbm import LGBMClassifier; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBClassifier; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm: top3['LightGBM'] = LGBMClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3['ExtraTrees'] = ExtraTreesClassifier(n_estimators=400, random_state=SEED, n_jobs=-1)
if has_xgb: top3['XGBoost'] = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, n_jobs=-1, eval_metric='logloss')
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3['AdaBoost'] = AdaBoostClassifier(n_estimators=200, random_state=SEED)
top3['GradBoosting'] = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=SEED)
print(f'Top 3: {list(top3.keys())}')


## 22. Final Training and Evaluation of Top 3


In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
final = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    final[name] = {'Accuracy': accuracy_score(y_test,yp),
                   'F1-macro': f1_score(y_test,yp,average='macro',zero_division=0),
                   'F1-weighted': f1_score(y_test,yp,average='weighted',zero_division=0)}
    print(f'\n{name}:'); print(classification_report(y_test,yp))
pd.DataFrame(final).T


In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(5*len(top3),4))
if len(top3)==1: axes=[axes]
for ax,(n,m) in zip(axes, top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, m.predict(X_te_t), ax=ax, cmap='Blues')
    ax.set_title(n)
plt.tight_layout(); plt.show()


In [ ]:
if y_test.nunique() == 2:
    fig, ax = plt.subplots(figsize=(8,5))
    for n,m in top3.items():
        RocCurveDisplay.from_estimator(m, X_te_t, y_test, ax=ax, name=n)
    ax.plot([0,1],[0,1],'k--',alpha=0.5); ax.set_title('ROC Curves')
    plt.tight_layout(); plt.show()
else:
    print('Multi-class: ROC per-class omitted for brevity.')


## 23. Error Analysis


In [ ]:
bm = list(top3.values())[0]; ypb = bm.predict(X_te_t)
print(f'Misclassified: {(y_test.values!=ypb).sum()}/{len(y_test)} ({(y_test.values!=ypb).mean():.1%})')
if y_test.nunique() == 2:
    fn = (y_test.values==1) & (ypb==0); fp = (y_test.values==0) & (ypb==1)
    print(f'FN (wrong condition): {fn.sum()}')
    print(f'FP (misclassified): {fp.sum()}')


## 24. Interpretation and Business Insight

- **Odometer** and **car age** are the strongest condition predictors
- **Price** correlates with condition but is not perfectly predictive
- **Miles per year** captures usage intensity
- Manufacturer and type provide context for expected condition


## 25. Limitations

1. Self-reported condition (subjective)
2. Missing values in many columns
3. Price may reflect condition (partial leakage concern)
4. No inspection data
5. Craigslist data quality varies


## 26. How to Improve This Project

1. Image-based condition assessment
2. VIN decode for accident history
3. Market price normalization
4. Regional condition patterns
5. NLP on listing descriptions


## 27. Production Considerations

- Automated listing grading
- Dealer inventory scoring
- Insurance risk pricing
- Buyer recommendation system
- Regular model updates with new listings


## 28. Common Mistakes

1. Including URL/description as features
2. Not handling missing odometer
3. Price as direct predictor (leakage risk)
4. Ignoring regional effects
5. Not encoding high-cardinality makes/models


## 29. Mini Challenge / Exercises

1. Binary: good vs poor condition
2. Only mileage + age — accuracy?
3. By manufacturer — do patterns differ?
4. Feature importance visualization
5. Price prediction (regression) instead


## 30. Final Summary / Key Takeaways

| Task | Multi-class car condition |
| Difficulty | Hard |
| Key | Odometer + age drive condition |
